# Edinburgh Airbnb Statistical Analysis

## Section 05 — Hypothesis Testing

This notebook tests five marketplace hypotheses using the Edinburgh Airbnb
warehouse created in Section 03.

For each test, the notebook records:
- Null and alternative hypotheses
- Test selection and rationale
- Test statistic, p-value, and effect size
- Business interpretation

In [ ]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy import stats


def cohens_d(group_a: pd.Series, group_b: pd.Series) -> float:
    """Calculate Cohen's d for two independent groups."""

    a = group_a.dropna().astype(float)
    b = group_b.dropna().astype(float)

    if len(a) < 2 or len(b) < 2:
        return np.nan

    pooled_std = np.sqrt(
        ((len(a) - 1) * a.std(ddof=1) ** 2 + (len(b) - 1) * b.std(ddof=1) ** 2)
        / (len(a) + len(b) - 2)
    )

    if pooled_std == 0:
        return np.nan

    return (a.mean() - b.mean()) / pooled_std


def eta_squared(f_statistic: float, df_between: int, df_within: int) -> float:
    """Calculate eta-squared from one-way ANOVA results."""

    return (f_statistic * df_between) / (f_statistic * df_between + df_within)


def summarize_test(
    hypothesis_id: str,
    test_name: str,
    statistic: float,
    p_value: float,
    effect_size: float,
    effect_label: str,
    group_a_label: str,
    group_b_label: str,
    group_a_mean: float,
    group_b_mean: float,
) -> pd.DataFrame:
    """Create a standard result table for one hypothesis test."""

    return pd.DataFrame(
        [
            {
                "hypothesis": hypothesis_id,
                "test": test_name,
                "group_a": group_a_label,
                "group_b": group_b_label,
                "group_a_mean": round(group_a_mean, 4),
                "group_b_mean": round(group_b_mean, 4),
                "statistic": round(statistic, 4),
                "p_value": p_value,
                "effect_size": round(effect_size, 4) if pd.notna(effect_size) else np.nan,
                "effect_metric": effect_label,
                "significant_0_05": p_value < 0.05,
            }
        ]
    )


project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

database_path = project_root / "database" / "airbnb.duckdb"
connection = duckdb.connect(str(database_path), read_only=True)

listings_query = """
SELECT
    f.listing_id,
    n.neighbourhood,
    f.price,
    f.room_type,
    f.number_of_reviews,
    f.review_scores_rating,
    h.host_is_superhost
FROM dw.fact_listing_performance f
LEFT JOIN dw.dim_neighbourhood n
    ON f.neighbourhood_key = n.neighbourhood_key
LEFT JOIN dw.dim_host h
    ON f.host_id = h.host_id
WHERE f.is_valid_price = TRUE
  AND f.is_valid_location = TRUE
"""

calendar_query = """
SELECT
    minimum_nights,
    is_weekend
FROM dw.fact_calendar
WHERE minimum_nights IS NOT NULL
"""

listings = connection.execute(listings_query).fetchdf()
calendar = connection.execute(calendar_query).fetchdf()

print(f"Listings used: {len(listings):,}")
print(f"Calendar rows used: {len(calendar):,}")
listings.head()

## H1: Entire-home listings command higher prices than private rooms

- **Null hypothesis:** There is no difference in mean price between entire-home/apartment listings and private-room listings.
- **Alternative hypothesis:** Entire-home/apartment listings have a higher mean price than private-room listings.
- **Test:** Welch's independent t-test (unequal variances expected for price data).
- **Assumptions:** Prices are numeric and independent across listings; normality is not strictly required at this sample size.

In [ ]:
entire_home = listings.loc[
    listings["room_type"] == "Entire home/apt", "price"
]
private_room = listings.loc[
    listings["room_type"] == "Private room", "price"
]

h1_stat, h1_p = stats.ttest_ind(entire_home, private_room, equal_var=False)
h1_effect = cohens_d(entire_home, private_room)

h1_results = summarize_test(
    hypothesis_id="H1",
    test_name="Welch t-test",
    statistic=h1_stat,
    p_value=h1_p,
    effect_size=h1_effect,
    effect_label="Cohen's d",
    group_a_label="Entire home/apt",
    group_b_label="Private room",
    group_a_mean=entire_home.mean(),
    group_b_mean=private_room.mean(),
)

display(h1_results)
print(f"Entire home mean price: £{entire_home.mean():.2f}")
print(f"Private room mean price: £{private_room.mean():.2f}")

### Business interpretation — H1

If the result is significant with a positive effect size, entire-home listings can
justify a pricing premium because guests pay for privacy and full-property access.
Private-room hosts should not benchmark against entire-home medians because the
products serve different guest segments.

## H2: Superhost listings achieve higher review scores than non-superhost listings

- **Null hypothesis:** There is no difference in mean review score between superhost and non-superhost listings.
- **Alternative hypothesis:** Superhost listings have a higher mean review score.
- **Test:** Welch's independent t-test on `review_scores_rating`.
- **Assumptions:** Review scores are treated as continuous and independently observed.

In [ ]:
reviewed = listings.dropna(subset=["review_scores_rating", "host_is_superhost"])

superhost_scores = reviewed.loc[
    reviewed["host_is_superhost"] == True, "review_scores_rating"
]
non_superhost_scores = reviewed.loc[
    reviewed["host_is_superhost"] == False, "review_scores_rating"
]

h2_stat, h2_p = stats.ttest_ind(superhost_scores, non_superhost_scores, equal_var=False)
h2_effect = cohens_d(superhost_scores, non_superhost_scores)

h2_results = summarize_test(
    hypothesis_id="H2",
    test_name="Welch t-test",
    statistic=h2_stat,
    p_value=h2_p,
    effect_size=h2_effect,
    effect_label="Cohen's d",
    group_a_label="Superhost",
    group_b_label="Non-superhost",
    group_a_mean=superhost_scores.mean(),
    group_b_mean=non_superhost_scores.mean(),
)

display(h2_results)
print(f"Superhost mean review score: {superhost_scores.mean():.3f}")
print(f"Non-superhost mean review score: {non_superhost_scores.mean():.3f}")

### Business interpretation — H2

A significant positive difference suggests the superhost badge is associated with
better guest satisfaction outcomes. Even if the statistical difference is small,
it may still matter commercially because guests use ratings as a trust signal when
choosing between similar listings.

## H3: Listings with more than 10 reviews have different prices than listings with fewer

- **Null hypothesis:** There is no difference in mean price between listings with more than 10 reviews and listings with 10 or fewer reviews.
- **Alternative hypothesis:** The two groups have different mean prices.
- **Test:** Welch's independent t-test.
- **Assumptions:** Review count split is treated as an independent group comparison.

In [ ]:
high_review_volume = listings.loc[listings["number_of_reviews"] > 10, "price"]
low_review_volume = listings.loc[listings["number_of_reviews"] <= 10, "price"]

h3_stat, h3_p = stats.ttest_ind(high_review_volume, low_review_volume, equal_var=False)
h3_effect = cohens_d(high_review_volume, low_review_volume)

h3_results = summarize_test(
    hypothesis_id="H3",
    test_name="Welch t-test",
    statistic=h3_stat,
    p_value=h3_p,
    effect_size=h3_effect,
    effect_label="Cohen's d",
    group_a_label="> 10 reviews",
    group_b_label="<= 10 reviews",
    group_a_mean=high_review_volume.mean(),
    group_b_mean=low_review_volume.mean(),
)

display(h3_results)
print(f"Mean price (>10 reviews): £{high_review_volume.mean():.2f}")
print(f"Mean price (<=10 reviews): £{low_review_volume.mean():.2f}")

### Business interpretation — H3

If established listings with more reviews command higher prices, review volume may
act as a credibility signal that supports premium positioning. Newer listings with
few reviews may need stronger pricing discipline or promotional strategy until they
accumulate enough social proof.

## H4: Neighbourhood average prices differ significantly

- **Null hypothesis:** Mean listing prices are equal across neighbourhood groups.
- **Alternative hypothesis:** At least one neighbourhood mean price differs.
- **Test:** One-way ANOVA across neighbourhoods with at least 15 listings.
- **Assumptions:** Groups are independent; large neighbourhood samples reduce sensitivity to normality violations.

In [ ]:
eligible_neighbourhoods = (
    listings.groupby("neighbourhood")
    .size()
    .loc[lambda s: s >= 15]
    .index
)

neighbourhood_groups = [
    group["price"].values
    for neighbourhood, group in listings[listings["neighbourhood"].isin(eligible_neighbourhoods)].groupby("neighbourhood")
]

h4_stat, h4_p = stats.f_oneway(*neighbourhood_groups)
df_between = len(neighbourhood_groups) - 1
df_within = sum(len(group) for group in neighbourhood_groups) - len(neighbourhood_groups)
h4_effect = eta_squared(h4_stat, df_between, df_within)

h4_results = pd.DataFrame(
    [
        {
            "hypothesis": "H4",
            "test": "One-way ANOVA",
            "groups_tested": len(neighbourhood_groups),
            "statistic": round(h4_stat, 4),
            "p_value": h4_p,
            "effect_size": round(h4_effect, 4),
            "effect_metric": "eta-squared",
            "significant_0_05": h4_p < 0.05,
        }
    ]
)

display(h4_results)

neighbourhood_means = (
    listings[listings["neighbourhood"].isin(eligible_neighbourhoods)]
    .groupby("neighbourhood")["price"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .round(2)
)

print("Top neighbourhood mean prices:")
display(neighbourhood_means)

### Business interpretation — H4

A significant ANOVA result confirms that location matters in Edinburgh's Airbnb
market beyond random noise. Pricing strategy should be neighbourhood-specific rather
than city-wide. Investors and hosts should benchmark against local competitors, not
the Edinburgh market average.

## H5: Weekend vs weekday calendar patterns differ significantly

- **Null hypothesis:** There is no difference in mean `minimum_nights` between weekend and weekday calendar records.
- **Alternative hypothesis:** Weekend and weekday calendar records have different mean `minimum_nights`.
- **Test:** Welch's independent t-test on calendar data.

**Data note:** The Edinburgh calendar file does not include daily price fields. This
test therefore uses `minimum_nights` from calendar data as a weekend demand/policy
signal, which still satisfies a calendar-based weekend vs weekday comparison.

In [ ]:
weekend_min_nights = calendar.loc[calendar["is_weekend"] == True, "minimum_nights"]
weekday_min_nights = calendar.loc[calendar["is_weekend"] == False, "minimum_nights"]

h5_stat, h5_p = stats.ttest_ind(weekend_min_nights, weekday_min_nights, equal_var=False)
h5_effect = cohens_d(weekend_min_nights, weekday_min_nights)

h5_results = summarize_test(
    hypothesis_id="H5",
    test_name="Welch t-test",
    statistic=h5_stat,
    p_value=h5_p,
    effect_size=h5_effect,
    effect_label="Cohen's d",
    group_a_label="Weekend minimum nights",
    group_b_label="Weekday minimum nights",
    group_a_mean=weekend_min_nights.mean(),
    group_b_mean=weekday_min_nights.mean(),
)

display(h5_results)
print(f"Weekend mean minimum nights: {weekend_min_nights.mean():.2f}")
print(f"Weekday mean minimum nights: {weekday_min_nights.mean():.2f}")

### Business interpretation — H5

If weekend minimum-night requirements differ from weekdays, hosts may be using stay
rules to manage demand during peak periods such as weekends, festivals, or holiday
travel windows. Revenue managers should review minimum-stay settings as part of
weekend strategy, especially when daily price fields are unavailable.

## Combined Results Summary

In [ ]:
all_results = pd.concat(
    [h1_results, h2_results, h3_results, h5_results],
    ignore_index=True,
)

summary_table = pd.concat(
    [
        all_results,
        h4_results.rename(
            columns={
                "groups_tested": "group_a",
            }
        ),
    ],
    ignore_index=True,
    sort=False,
)

summary_output = project_root / "reports" / "tables" / "hypothesis_test_summary.csv"
summary_output.parent.mkdir(parents=True, exist_ok=True)
summary_table.to_csv(summary_output, index=False)

display(summary_table)
print(f"Saved: {summary_output}")

## Section 05 Summary

This notebook completed the core statistical testing requirements for Edinburgh:

- H1: Entire home vs private room price difference
- H2: Superhost vs non-superhost review scores
- H3: High-review vs low-review listing prices
- H4: Neighbourhood price differences using ANOVA
- H5: Weekend vs weekday calendar differences using minimum nights

Results were exported to `reports/tables/hypothesis_test_summary.csv` for use in the
final assignment report.